In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.linalg import expm

# ---------- Paths ----------
def find_base_dir():
    env = os.environ.get("PROJECT_DIR")
    if env and (Path(env) / "data" / "raw").exists():
        return Path(env)
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists():
            return p
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(title="Select the project folder (the one containing data/raw)")
        root.destroy()
    except Exception:
        chosen = input("Paste the project folder path: ").strip().strip('"')
    if chosen and (Path(chosen) / "data" / "raw").exists():
        return Path(chosen)
    raise FileNotFoundError("data/raw not found in the selected folder.")

BASE_DIR = find_base_dir()
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

def safe_save(save_fn, filename):
    try:
        save_fn(OUT_DIR / filename)
        print("Saved:", OUT_DIR / filename)
    except OSError as e:
        fallback = Path.home() / "part3b_outputs"
        fallback.mkdir(exist_ok=True)
        save_fn(fallback / filename)
        print("Could not write to", OUT_DIR, "-", e)
        print("Saved to", fallback / filename, "instead.")

# ---------- Data ----------
ins = pd.read_csv(RAW_DIR / "insured_transitions.csv")

unique_origin = ins.drop_duplicates(subset=["district_id", "origin"])[
    ["district_id", "origin", "exposure_py", "initial_lives"]
]
pooled_exp = unique_origin.groupby("origin")["exposure_py"].sum().to_dict()
pooled_jumps = ins.groupby(["origin", "destination"])["transition_count"].sum().to_dict()

lam_HI = pooled_jumps[("H", "I")] / pooled_exp["H"]
lam_HD = pooled_jumps[("H", "D")] / pooled_exp["H"]
lam_IH = pooled_jumps[("I", "H")] / pooled_exp["I"]
lam_ID = pooled_jumps[("I", "D")] / pooled_exp["I"]

Q = np.array([
    [-(lam_HI + lam_HD),  lam_HI,            lam_HD],
    [ lam_IH,            -(lam_IH + lam_ID),  lam_ID],
    [ 0.0,                0.0,               0.0]
], dtype=float)

# 1. Mean holding times (in years) for living states H and I
holding_df = pd.DataFrame([
    {"state": "H", "exit_rate_per_yr": -Q[0, 0], "mean_holding_time_yrs": 1.0 / (-Q[0, 0]), "mean_holding_time_months": (1.0 / (-Q[0, 0])) * 12.0},
    {"state": "I", "exit_rate_per_yr": -Q[1, 1], "mean_holding_time_yrs": 1.0 / (-Q[1, 1]), "mean_holding_time_months": (1.0 / (-Q[1, 1])) * 12.0},
])

# 2. Next-jump probabilities from Ill (I)
p_jump_IH = lam_IH / (-Q[1, 1])
p_jump_ID = lam_ID / (-Q[1, 1])
jumps_from_I_df = pd.DataFrame([
    {"origin": "I", "next_destination": "H (recovery)", "intensity": lam_IH, "next_jump_prob": p_jump_IH},
    {"origin": "I", "next_destination": "D (death)",    "intensity": lam_ID, "next_jump_prob": p_jump_ID},
])

# 3. Compare with 1-year occupancy transition P_IH(1)
P1 = expm(Q * 1.0)

print("\n=== MEAN HOLDING TIMES ===")
display(holding_df)

print("\n=== NEXT-JUMP PROBABILITIES FROM ILL (I) ===")
display(jumps_from_I_df)

print(f"\nNext-jump p(I -> H) = {p_jump_IH:.6f} vs 1-year occupancy P_IH(1) = {P1[1, 0]:.6f}")

safe_save(lambda p: holding_df.to_csv(p, index=False), "part3b_holding_times.csv")
safe_save(lambda p: jumps_from_I_df.to_csv(p, index=False), "part3b_next_jump_from_ill.csv")

BASE_DIR: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-

=== MEAN HOLDING TIMES ===


,state,exit_rate_per_yr,mean_holding_time_yrs,mean_holding_time_months
0,H,0.121514,8.229535,98.754424
1,I,0.312340,3.201643,38.419720



=== NEXT-JUMP PROBABILITIES FROM ILL (I) ===


,origin,next_destination,intensity,next_jump_prob
0,I,H (recovery),0.245058,0.784588
1,I,D (death),0.067282,0.215412



Next-jump p(I -> H) = 0.784588 vs 1-year occupancy P_IH(1) = 0.198404
Saved: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part3b_holding_times.csv
Saved: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part3b_next_jump_from_ill.csv
